# Classificador 

In [1]:
import pandas as pd

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

In [3]:
import numpy as np

In [4]:
from string import punctuation

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
import random

In [8]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

In [9]:
df = pd.read_csv("d:/git/dados/nlp/news_sentiment_analysis.csv", encoding="utf-8")

In [10]:
df = df.drop(columns=["Source", "Author", "URL", "Published At", "Sentiment", "Title"])

In [11]:
df

,Description,Type
0,"ST. GEORGE — Kaitlyn Larson, a first-year teac...",Business
1,"Harare, Zimbabwe – Local businesses are grappl...",Business
2,(marketscreener.com) Billionaire Elon Musk has...,Business
3,(marketscreener.com) A U.S. trade regulator on...,Business
4,4.5 million households in the U.S. have solar ...,Business
...,...,...
3495,QRG Capital Management Inc. increased its stak...,Technology
3496,QRG Capital Management Inc. bought a new posit...,Technology
3497,QRG Capital Management Inc. boosted its stake ...,Technology
3498,"WESTFORD, Mass., July 18, 2024 /PRNewswire/ --...",Technology


In [12]:
print(df["Type"].unique())
print(df["Type"].describe())

['Business' 'Entertainment' 'General' 'Health' 'Science' 'Sports'
 'Technology']
count         3500
unique           7
top       Business
freq           500
Name: Type, dtype: object


In [14]:
# mapping = {'Business': 1, 'Entertainment': 2, 'General': 3, 'Health': 4, 'Science':5, 'Sports':6, 'Technology': 7}
# mapping = {
#         'Business': [1, 0, 0, 0, 0, 0, 0], 
#         'Entertainment': [0, 1, 0, 0, 0, 0, 0],
#         'General': [0, 0, 1, 0, 0, 0, 0],
#         'Health': [0, 0, 0, 1, 0, 0, 0],
#         'Science': [0, 0, 0, 0, 1, 0, 0],
#         'Sports': [0, 0, 0, 0, 0, 1, 0],
#         'Technology': [0, 0, 0, 0, 0, 0, 1]
# }
# df["sentiment_number"] = df["Sentiment"].map( mapping ) 

In [29]:
MAX_CLASSES = 7

In [15]:
encoder = LabelEncoder()
hot_encoder = OneHotEncoder(max_categories=MAX_CLASSES, sparse_output=False)

In [16]:
Y_labels = encoder.fit_transform(df["Type"])
Y_labels = Y_labels.reshape(-1, 1)

In [17]:
Y = torch.tensor(hot_encoder.fit_transform(Y_labels), dtype=torch.float32)

In [18]:
table = str.maketrans("", "", punctuation)

# def limpar( texto ):
#     texto_limpo = texto.lower().translate(table)
#     return texto_limpo

def limpar( texto ):
    return texto

In [19]:
df["description_clean"] = df["Description"].apply(limpar)

In [38]:
MAX_PALAVRAS = 500

In [53]:
dicionario = {
    "<UNKNOWN>": 0,
    "<PAD>": 1
}
contador_palavras = len(dicionario.keys())
contador_palavras

2

In [54]:
lista_numeros = []
for texto in df["description_clean"]:
    numeros = []
    for palavra in texto.split(" "):
        if palavra not in dicionario: 
            dicionario[palavra] = contador_palavras
            contador_palavras += 1
        numeros.append(dicionario.get(palavra, 0))
    lista_numeros.append(numeros)

In [57]:
# Identificar qual frase tem a maior quantidade de palavras
max_size = 0
for frase in lista_numeros:
    if len(frase) > max_size:
        max_size = len(frase)
max_size        

109

In [59]:
lista_padded = []
for frase in lista_numeros:
    number_pads = max_size - len(frase)
    frase_padded = []
    for i in range(number_pads):
        frase_padded.append( dicionario["<PAD>"] )
    frase_padded.extend( frase )
    lista_padded.append( frase_padded )

In [ ]:
### 
# [1 , 1 , 1 , 1 , 12, 17, 19, 10]
# [12, 17, 19, 10, 34, 78, 23, 19]
# [1 , 1 , 1 , 1 , 1 , 12, 17, 19]

In [62]:
len(lista_padded[0])

109

In [61]:
X = torch.tensor(lista_padded, dtype=torch.int32)
X

tensor([[    1,     1,     1,  ...,    48,    49,    50],
        [    1,     1,     1,  ...,    45,    63,    64],
        [    1,     1,     1,  ...,   101,   102,   103],
        ...,
        [    1,     1,     1,  ..., 27429,  1080,    50],
        [    1,     1,     1,  ..., 29404,    19, 29406],
        [    1,     1,     1,  ...,  1697, 12784,    50]], dtype=torch.int32)

In [63]:
print("X: ", X.dtype, X.shape, X.ndim)
print("Y: ", Y.dtype, Y.shape, Y.ndim)

X:  torch.int32 torch.Size([3500, 109]) 2
Y:  torch.float32 torch.Size([3500, 7]) 2


In [26]:
X_treino, X_teste, Y_treino, Y_teste = train_test_split(X, Y, test_size=0.2, random_state=100)

In [27]:
len(X_treino)
# X_treino[0].sum()

2800

In [30]:
modelo = nn.Linear(in_features = MAX_PALAVRAS, out_features = MAX_CLASSES)
# modelo = nn.Sequential(
#     nn.Linear(in_features = MAX_PALAVRAS, out_features=1),
#     nn.Sigmoid()
# )


In [31]:
criterio = nn.CrossEntropyLoss() # Cross Entropy Loss
otimizador = optim.SGD( modelo.parameters(), lr=0.01 )

In [32]:
for epoca in range(1, 3000):
    Y_hat = modelo( X_treino )
    loss = criterio( Y_hat, Y_treino )
    otimizador.zero_grad()
    loss.backward()
    otimizador.step()
    if epoca % 100 == 0:
        print(f"Epoca: {epoca}\tLoss:{loss}")

Epoca: 100	Loss:1.7620786428451538
Epoca: 200	Loss:1.6182533502578735
Epoca: 300	Loss:1.5041277408599854
Epoca: 400	Loss:1.4091620445251465
Epoca: 500	Loss:1.3280675411224365
Epoca: 600	Loss:1.257668137550354
Epoca: 700	Loss:1.1958327293395996
Epoca: 800	Loss:1.1410236358642578
Epoca: 900	Loss:1.0920768976211548
Epoca: 1000	Loss:1.0480839014053345
Epoca: 1100	Loss:1.008319616317749
Epoca: 1200	Loss:0.9721961617469788
Epoca: 1300	Loss:0.9392315745353699
Epoca: 1400	Loss:0.9090257287025452
Epoca: 1500	Loss:0.8812441825866699
Epoca: 1600	Loss:0.8556042313575745
Epoca: 1700	Loss:0.8318660259246826
Epoca: 1800	Loss:0.8098238706588745
Epoca: 1900	Loss:0.7893008589744568
Epoca: 2000	Loss:0.7701436281204224
Epoca: 2100	Loss:0.7522187829017639
Epoca: 2200	Loss:0.7354096174240112
Epoca: 2300	Loss:0.7196137309074402
Epoca: 2400	Loss:0.7047407031059265
Epoca: 2500	Loss:0.6907104253768921
Epoca: 2600	Loss:0.6774519085884094
Epoca: 2700	Loss:0.6649019122123718
Epoca: 2800	Loss:0.6530036330223083
Epo

In [33]:
vetorizador_predict = CountVectorizer(max_features = MAX_PALAVRAS, vocabulary=dicionario)

In [34]:
indice = random.randint(0, 3500)
predict_set = [ df["description_clean"][indice] ]
tipo = [ df["Type"][indice] ]
print(f"Indice: {indice}\t\tTipo: {tipo}")
print(predict_set)

# predict_set = [
#     # "the fruitwatch initiative a groundbreaking citizen science project has significantly enhanced the accuracy of predicting flowering times for fruit trees across great britain this improvement is vital for the agricultural sector enabling better planning for pest management and pollinator support which are crucial for maintaining optimal fruit yield and quality"
#     # "researchers at the institute for systems biology in seattle found that bowel movement frequency could predict kidney and liver damage as well as mental health issues like depression"
#     # "ap entertainment writer new york ap — richard simmons television8217s hyperactive court jester of physical fitness who built a miniempire in his trademark tank tops and short shorts by urging the overweight to exercise and eat better died saturday he turned 76 on friday los angeles police and fire departments say they responded to athe post richard simmons a fitness guru who mixed laughs and sweat dies at 76 appeared first on kvia"
# ]

Indice: 2587		Tipo: ['Science']
['Ellina Mhlanga Senior Sports Reporter FOR Olympian and now athletics coach, Cuthbert Nyasango, Tapiwanashe Makarawu’s qualification for the forthcoming Summer Games is something he says he had always expected given the sprinter’s “huge’’ potential. Seven athletes including Makarawu, a former National Sports Academy athlete and Bindura University of Science Education student, will represent Zimbabwe at [&#8230;]']


In [37]:
X_predict = vetorizador_predict.fit_transform( predict_set )
# list_reverse = ['Negative', 'Positive']
with torch.no_grad():
    X_pred = torch.tensor(X_predict.toarray(), dtype=torch.float32)
    resposta = modelo( X_pred )
    indice = np.argmax(resposta)
    tipo = encoder.inverse_transform( [indice] )
    print(f"Resposta: {resposta}\tIndice: {indice}\tTipo: {tipo}")
    # indice = round( resposta.item() )
    # print(f"Este texto é sobre {resposta} {indice} {list_reverse[indice]}")
# X_predict.toarray()

Resposta: tensor([[-0.7014, -0.8505, -0.9961,  1.7249,  2.6732, -0.8582, -1.1119]])	Indice: 4	Tipo: ['Science']
